In [14]:
!pip install lightgbm

import os
import pandas as pd
import numpy as np

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

ML2_PATH = "../dataset/output/ml2_features"
HOURLY_PATH = "../dataset/output/hourly_grid_summary"

features_spark = spark.read.parquet(ML2_PATH).select(
    "grid_id",
    "feature_timestamp",
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
)

hourly_spark = (
    spark.read.parquet(HOURLY_PATH)
    .select(
        "grid_id",
        "timestamp",
        "total_activity"
    )
    .withColumn(
        "timestamp",
        F.col("timestamp").cast("timestamp")
    )
)

features_df = (
    features_spark
    .toPandas()
)

hourly_df = (
    hourly_spark
    .toPandas()
)

features_df["feature_timestamp"] = pd.to_datetime(
    features_df["feature_timestamp"]
)

hourly_df["timestamp"] = pd.to_datetime(
    hourly_df["timestamp"]
)

features_df = features_df.sort_values(
    ["feature_timestamp", "grid_id"]
).reset_index(drop=True)

hourly_df = hourly_df.sort_values(
    ["grid_id", "timestamp"]
).reset_index(drop=True)

print("ML2 rows:", len(features_df))
print("Hourly rows:", len(hourly_df))

c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\krishnarajan.r\AppData\Local\Programs\Python\Python312\Lib\site-packa

ML2 rows: 1209994
Hourly rows: 1679994


In [15]:
# ---------------------------------------------------------
# 24-hour historical baseline per grid
# Only information BEFORE t is used.
# ---------------------------------------------------------

hourly_df["baseline_24h"] = (
    hourly_df
    .groupby("grid_id")["total_activity"]
    .transform(
        lambda s: s.shift(1).rolling(
            window=24,
            min_periods=6
        ).median()
    )
)

# ---------------------------------------------------------
# Actual next-hour activity
# This is the FUTURE TARGET information.
# It must never be used as a model feature.
# ---------------------------------------------------------

hourly_df["future_activity_t1"] = (
    hourly_df
    .groupby("grid_id")["total_activity"]
    .shift(-1)
)

# ---------------------------------------------------------
# Create target
#
# Future activity >= 1.5 × baseline known at t
# ---------------------------------------------------------

valid_target = (
    hourly_df["baseline_24h"].notna()
    &
    (hourly_df["baseline_24h"] > 0)
    &
    hourly_df["future_activity_t1"].notna()
)

hourly_df["target"] = 0

hourly_df.loc[
    valid_target
    &
    (
        hourly_df["future_activity_t1"]
        >=
        hourly_df["baseline_24h"] * 1.5
    ),
    "target"
] = 1

print(
    hourly_df[
        [
            "grid_id",
            "timestamp",
            "total_activity",
            "baseline_24h",
            "future_activity_t1",
            "target"
        ]
    ].head(30)
)

    grid_id           timestamp  total_activity  baseline_24h  \
0         1 2013-11-01 00:00:00         62.0092           NaN   
1         1 2013-11-01 01:00:00         46.3654           NaN   
2         1 2013-11-01 02:00:00         42.0870           NaN   
3         1 2013-11-01 03:00:00         35.0978           NaN   
4         1 2013-11-01 04:00:00         32.2741           NaN   
5         1 2013-11-01 05:00:00         35.4160           NaN   
6         1 2013-11-01 06:00:00         36.1216      38.75150   
7         1 2013-11-01 07:00:00         44.7950      36.12160   
8         1 2013-11-01 08:00:00         67.4200      39.10430   
9         1 2013-11-01 09:00:00         83.7437      42.08700   
10        1 2013-11-01 10:00:00        100.7637      43.44100   
11        1 2013-11-01 11:00:00         95.1478      44.79500   
12        1 2013-11-01 12:00:00         91.2744      45.58020   
13        1 2013-11-01 13:00:00         92.0887      46.36540   
14        1 2013-11-01 14

In [16]:
target_df = hourly_df[
    [
        "grid_id",
        "timestamp",
        "baseline_24h",
        "future_activity_t1",
        "target"
    ]
].copy()

ml3_df = features_df.merge(
    target_df,
    left_on=[
        "grid_id",
        "feature_timestamp"
    ],
    right_on=[
        "grid_id",
        "timestamp"
    ],
    how="inner"
)

ml3_df = ml3_df[
    ml3_df["baseline_24h"].notna()
    &
    ml3_df["future_activity_t1"].notna()
    &
    (ml3_df["baseline_24h"] > 0)
].copy()

ml3_df = ml3_df.sort_values(
    "feature_timestamp"
).reset_index(drop=True)

print("ML3 rows:", len(ml3_df))

print("\nClass balance:")
print(
    ml3_df["target"].value_counts()
)

print("\nClass proportion:")
print(
    ml3_df["target"].value_counts(
        normalize=True
    )
)

ML3 rows: 1199994

Class balance:
target
0    1076384
1     123610
Name: count, dtype: int64

Class proportion:
target
0    0.896991
1    0.103009
Name: proportion, dtype: float64


In [18]:
ml3_df["date"] = (
    ml3_df["feature_timestamp"]
    .dt.normalize()
)

unique_dates = sorted(
    ml3_df["date"].dropna().unique()
)

print("Available dates:")
for d in unique_dates:
    print(pd.Timestamp(d).date())

if len(unique_dates) != 6:
    raise ValueError(
        f"Expected 6 dates, found {len(unique_dates)}"
    )

train_dates = unique_dates[:4]
test_dates = unique_dates[4:]

train_df = ml3_df[
    ml3_df["date"].isin(train_dates)
].copy()

test_df = ml3_df[
    ml3_df["date"].isin(test_dates)
].copy()

print("\nTRAIN")
print(
    train_df["feature_timestamp"].min(),
    "to",
    train_df["feature_timestamp"].max()
)

print("Training days:", [
    pd.Timestamp(d).date()
    for d in train_dates
])

print("\nTEST")
print(
    test_df["feature_timestamp"].min(),
    "to",
    test_df["feature_timestamp"].max()
)

print("Testing days:", [
    pd.Timestamp(d).date()
    for d in test_dates
])

print("\nTrain rows:", len(train_df))
print("Test rows :", len(test_df))

print("\nTrain base rate:")
print(train_df["target"].mean())

print("\nTest base rate:")
print(test_df["target"].mean())

Available dates:
2013-11-02
2013-11-03
2013-11-04
2013-11-05
2013-11-06
2013-11-07

TRAIN
2013-11-02 23:00:00 to 2013-11-05 23:00:00
Training days: [datetime.date(2013, 11, 2), datetime.date(2013, 11, 3), datetime.date(2013, 11, 4), datetime.date(2013, 11, 5)]

TEST
2013-11-06 00:00:00 to 2013-11-07 22:00:00
Testing days: [datetime.date(2013, 11, 6), datetime.date(2013, 11, 7)]

Train rows: 729997
Test rows : 469997

Train base rate:
0.11493060930387385

Test base rate:
0.08449202867252344


In [19]:
FEATURES = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share"
]

X_train = train_df[FEATURES]
y_train = train_df["target"]

X_test = test_df[FEATURES]
y_test = test_df["target"]

positive = (y_train == 1).sum()
negative = (y_train == 0).sum()

scale_pos_weight = (
    negative / positive
)

print("Positive samples:", positive)
print("Negative samples:", negative)
print("Scale positive weight:", scale_pos_weight)

model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=6,
    min_child_samples=50,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(
    X_train,
    y_train
)

test_probability = model.predict_proba(
    X_test
)[:, 1]

print("LightGBM training completed")

Positive samples: 83899
Negative samples: 646098
Scale positive weight: 7.700902275354891
LightGBM training completed


In [20]:
THRESHOLD = 0.50

test_prediction = (
    test_probability >= THRESHOLD
).astype(int)

accuracy = accuracy_score(
    y_test,
    test_prediction
)

precision = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

recall = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

f1 = f1_score(
    y_test,
    test_prediction,
    zero_division=0
)

print("\n================================")
print("ML3 FINAL LIGHTGBM")
print("================================")

print("Train period:")
print(
    train_df["feature_timestamp"].min(),
    "to",
    train_df["feature_timestamp"].max()
)

print("\nTest period:")
print(
    test_df["feature_timestamp"].min(),
    "to",
    test_df["feature_timestamp"].max()
)

print("\nThreshold:", THRESHOLD)
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1       :", round(f1, 4))
print("Base rate:", round(y_test.mean(), 4))

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_prediction,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        test_prediction
    )
)

# ---------------------------------------------------------
# Feature importance
# ---------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance_df)

# ---------------------------------------------------------
# Final prediction table
# ---------------------------------------------------------

test_results = test_df[
    [
        "grid_id",
        "feature_timestamp",
        "avg_activity",
        "activity_growth",
        "active_hours",
        "peak_ratio",
        "variability",
        "internet_share",
        "baseline_24h",
        "future_activity_t1",
        "target"
    ]
].copy()

test_results["risk_score"] = test_probability

test_results["prediction"] = test_prediction

test_results["prediction_label"] = np.where(
    test_prediction == 1,
    "RISK",
    "NO RISK"
)

print("\nSample predictions:")
print(
    test_results.head(20)
)


ML3 FINAL LIGHTGBM
Train period:
2013-11-02 23:00:00 to 2013-11-05 23:00:00

Test period:
2013-11-06 00:00:00 to 2013-11-07 22:00:00

Threshold: 0.5
Accuracy : 0.7713
Precision: 0.2159
Recall   : 0.6481
F1       : 0.3239
Base rate: 0.0845

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.78      0.86    430286
           1       0.22      0.65      0.32     39711

    accuracy                           0.77    469997
   macro avg       0.59      0.72      0.59    469997
weighted avg       0.90      0.77      0.82    469997


Confusion Matrix:
[[336793  93493]
 [ 13974  25737]]

Feature Importance:
           feature  importance
4      variability        1923
0     avg_activity        1905
1  activity_growth        1854
5   internet_share        1841
3       peak_ratio        1477
2     active_hours           0

Sample predictions:
        grid_id feature_timestamp  avg_activity  activity_growth  \
729997     6662        2013-